# nb01 — 신규 KIM 격자 준비와 8지점 변환

2026-07-01 정책 변경으로 도입된 신규 표준 API(`nph-kim_nc_xy_txt2_std`)는 격자 X/Y 로 지점을 지정한다.
이 노트북은 세 모델의 격자를 확보하고, 운영 8지점(육지 5 + 제주 3)의 X/Y 변환표를 만든다.

| 모델 | nwp | 격자 | 해상도 | 투영 |
|---|---|---|---|---|
| 전구 | NE57 | g576 (4320×2160) | 약 8 km | 위경도 등간격 |
| 지역 | R030 | r030 (1049×839) | 약 3 km | Lambert |
| 국지 | L010 | l010 (1535×1175) | 약 1.3 km | Lambert |

**확인된 함정**: 격자 조회 API(`nph-nwp_latlon_api`, disp=B)는 앞 4바이트가 (행수, 열수) short 2개이고,
돌려주는 격자가 자료 격자보다 행·열이 1 큰 **셀 모서리** 격자다. 4모서리 평균 = 자료 격자(셀 중심)와
1e-5도 수준으로 일치함을 r030 보유 nc 파일로 검증했다. l010 은 이 방법으로 중심 격자를 만들어
`grids/kim_l010_latlon.npz` 로 저장했다.

In [1]:
import sys
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
import numpy as np
import pandas as pd
import probe_lib as pl
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

def std_val(grp, nwp, nm, tmfc, hf, x, y, data="U", level=None):
    """캐시된 신규 std 응답에서 첫 값 (캐시에 없으면 실호출 1회)."""
    b = pl.fetch_std(grp, nwp, nm, tmfc, hf, x, y, data=data, level=level)
    if not b or "ERROR" in b:
        return None
    v = [float(t) for l in b.splitlines()
         if l.strip() and not l.startswith("#") for t in l.split()]
    return v[0] if v else None

TMFC = "2026070312"   # 본 연구의 기준 발표 (2026-07-03 12z) -- 캐시 고정
print("호출 예산 상태:", pl.budget_status())

호출 예산 상태: {'real_calls': 1901, 'hard_cap': 10000, 'remaining': 8099}


In [2]:
# 세 격자 로드 (g576/r030 = 제공받은 nc, l010 = API 바이너리 -> 셀 중심 변환 저장본)
grids = {}
for g in ["g576", "r030", "l010"]:
    la, lo = pl.load_grid(g)
    grids[g] = (la, lo)
    print(f"{g}: {la.shape}  위도 {la.min():.2f}~{la.max():.2f}  경도 {lo.min():.2f}~{lo.max():.2f}")

g576: (2160, 4320)  위도 -89.96~89.96  경도 0.00~359.92
r030: (839, 1049)  위도 25.27~49.69  경도 103.99~148.02
l010: (1535, 1175)  위도 30.78~45.13  경도 118.27~133.73


In [3]:
# 검증: r030 모서리 격자(API) 4점 평균 == 셀 중심(nc)
body = pl.fetch(pl.URL_LATLON, {"nwp": "r030", "latlon": "lat", "disp": "B"}, binary=True)
a, b_ = np.frombuffer(body[:4], dtype="<i2")
corner = np.frombuffer(body[4:], dtype="<f4").reshape(a, b_)
center = pl.corners_to_centers(corner)
print("모서리 격자:", corner.shape, "-> 중심:", center.shape)
print("보유 nc 와 최대 차이(도):", float(np.abs(center - grids["r030"][0]).max()))

모서리 격자: (840, 1050) -> 중심: (839, 1049)
보유 nc 와 최대 차이(도): 1.52587890625e-05


In [4]:
# 8지점 x 3격자 변환표 + 최근접 격자 이동거리(km)
rows = []
for p in pl.POINTS_ALL:
    r = {"지점": p["name"], "lat": p["lat"], "lon": p["lon"]}
    for g, (la, lo) in grids.items():
        x, y, dkm = pl.find_xy(p["lat"], p["lon"], la, lo)
        r[f"{g}_X"], r[f"{g}_Y"], r[f"{g}_km"] = x, y, round(dkm, 2)
    rows.append(r)
xy = pd.DataFrame(rows)
xy.to_csv("results/grid_xy_table.csv", index=False, encoding="utf-8-sig")
xy

,지점,lat,lon,g576_X,g576_Y,g576_km,r030_X,r030_Y,r030_km,l010_X,l010_Y,l010_km
0,Daegwallyeong(100),37.6772,128.7185,1546,1533,4.38,602,410,2.11,821,737,0.37
1,Wonju(114),37.3376,127.9466,1536,1529,4.87,581,397,1.52,756,698,0.47
2,Seosan(129),36.7766,126.4939,1519,1522,1.70,539,376,1.06,631,635,0.36
3,Pohang(138),36.0327,129.3799,1554,1513,3.43,624,351,0.81,885,561,0.50
4,Yeonggwang(252),35.2807,126.4750,1519,1504,2.55,539,321,1.35,630,472,0.48
5,solar_farm(south),33.3284,126.8366,1523,1480,4.14,550,250,1.53,664,259,0.53
6,West(Gosan),33.4427,126.1713,1515,1482,1.73,530,254,1.25,603,271,0.62
7,East(Seongsan),33.3868,126.8802,1524,1481,3.64,552,253,1.81,668,266,0.47


**판정**
- 국지(l010) 도메인은 위도 30.8~45.1, 경도 118.3~133.7 — 육지 5지점(최동단 포항 129.4E, 최북단 대관령 37.7N)을 전부 덮는다. 최근접 격자 거리도 최대 0.62 km 로 셀 반경 안.
- 전구 최근접 거리 ≤ 4.9 km(8 km 격자), 지역 ≤ 2.1 km(3 km 격자) — 모두 정상 범위.
- 실전 검증: 구 pt 엔드포인트에 위경도를 주면 응답 헤더에 X/Y 를 돌려주는데, 우리 변환값과 정확히 일치했다(nb02 참조).